### Merging Fixation, Recall, and Recognition Information on the basis of the billboards for all subjects

In [1]:
import pandas as pd
import glob
from tqdm import tqdm

### Reading in the recall df

In [2]:
memory_df = pd.read_csv('../data/01_interview_data/Recall_sheet_vr_driving_city.csv')
memory_df.fillna(0, inplace=True)
print(len(memory_df))
memory_df.tail()

43


,subject,ad_brunch,ad_burger,ad_coffee,ad_pizza,ad_furniture,ad_hotel,ad_lawyer,ad_smartphone,ad_fundraising,...,health_jogging,health_getsleep,health_doctor,health_caffeine,health_greatlake,health_mentalhealth,health_recycling,health_polarbears,condition,Note
38,sub039,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Count_Sales_Signs,0.0
39,sub040,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,Free_Viewing,0.0
40,sub041,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,Free_Viewing,0.0
41,sub042,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Free_Viewing,0.0
42,sub043,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,Count_Sales_Signs,0.0


### Reading in the recognition df

In [3]:
recognition_df = pd.read_csv('../data/02_qualtrics_data/recognition_survey_vr_city_cleaned.csv')
recognition_df.tail()
print(len(recognition_df))

43


In [4]:
# dropping unnecessary information from the rest of the survey (non-recognition)
recognition_clean_df = recognition_df.drop(['Age', 'Gender', 'Virtual_Sickness_1_discomfort',
       'Virtual_Sickness_2_fatigue', 'Virtual_Sickness_3_eyestrain',
       'Virtual_Sickness_4_difficultyfocusing', 'Virtual_Sickness_5_headache',
       'Virtual_Sickness_6_fullnessofhead', 'Virtual_Sickness_7_blurredvision',
       'Virtual_Sickness_8_dizzy', 'Virtual_Sickness_9_vertigo',
       'Spatial_Presence_1_transported', 'Spatial_Presence_2_threedimensional',
       'Spatial_Presence_3_realisticproportions',
       'Spatial_Presence_4_movefreely', 'Spatial_Presence_5_physicallypresent', 
       'dis_1_diamond_r', 'dis_1_diamond_c', 
       'dis_2_gym_r', 'dis_2_gym_c',
       'dis_3_donutpromo_r', 'dis_3_donutpromo_c',                             
       'dis_4_apple_r','dis_4_apple_c'], axis=1)

recognition_clean_df['Subject'] = recognition_clean_df['Subject'].str.lower()
print(len(recognition_clean_df))
recognition_clean_df.tail()

43


,Subject,ad_brunch,ad_brunch_c,ad_burger,ad_burger_c,ad_coffee,ad_coffee_c,ad_pizza,ad_pizza_c,ad_furniture,...,health_healthydiet_a,health_employee_a,health_jogging_a,health_getsleep_a,health_doctor_a,health_caffeine_a,health_greatlake_a,health_mentalhealt_a,health_recycling_a,health_polarbears_a
38,sub040,0,NaN,0,NaN,1,6.0,0,NaN,1,...,1,1,1,1,1,1,1,1,1,1
39,sub041,1,5.0,1,7.0,0,NaN,1,7.0,0,...,1,1,1,1,1,1,1,1,2,2
40,sub042,1,4.0,0,NaN,1,4.0,0,NaN,0,...,1,1,1,1,1,1,2,2,2,1
41,sub027,1,6.0,0,NaN,1,1.0,0,NaN,1,...,1,1,1,1,1,1,1,1,1,1
42,sub043,0,NaN,1,3.0,1,5.0,1,5.0,0,...,1,1,1,1,1,1,1,2,1,1


### Loop over all subjects and merge the recall, recognition, and fixation information

In [5]:
folders = glob.glob("../data/00_gaze_data/sub*")
folders.sort()
subjs = []
for f in folders:
    subjs.append(f[-6:])
print(len(subjs))  

43


In [6]:
#Code/data check

i=25
curr_sub_fixation_file = '../data/00_gaze_data/fixation_billboards/'  + subjs[i] +'_fixations_billboards.csv'
out_file_name = '../data/03_fixation_billboards_recall_recognition/'  + subjs[i] +'_fixations_billboards_recall_recognition_integrated.csv'

curr_sub_fix_df = pd.read_csv(curr_sub_fixation_file)
curr_sub_fix_df['billboard_image_at_position_text_attention'] = curr_sub_fix_df.loc[:, 'billboard_image_at_position_text']
curr_sub_fix_df['billboard_image_at_position_text'] = curr_sub_fix_df['billboard_image_at_position_text'].replace(to_replace=r'moreatt_', value='', regex=True)
curr_sub_fix_df['billboard_image_at_position_text'] = curr_sub_fix_df['billboard_image_at_position_text'].replace(to_replace=r'lessatt_', value='', regex=True)
curr_sub_fix_df.tail()

position_billboard_fix = curr_sub_fix_df[["position_index", "billboard_image_at_position_text", "billboard_image_at_position_text_attention", "fix_bill", "billboard#_fixated_count", "billboard#_duration"]]
position_billboard_fix.tail()

current_subject_recall = memory_df[memory_df['subject']==subjs[i]]
current_subject_recall = current_subject_recall.drop('subject', axis= 1)
current_subject_recall = current_subject_recall.drop('condition', axis= 1)
current_subject_recall = current_subject_recall.T
current_subject_recall['billboard'] = current_subject_recall.index
current_subject_recall.rename( columns={i :'recall'}, inplace=True )
current_subject_recall = current_subject_recall.reset_index(drop=True)
current_subject_recall.tail()

# #merge fixation and recall df
position_billboard_fix_recall = position_billboard_fix.merge(current_subject_recall, left_on = 'billboard_image_at_position_text', right_on = 'billboard')
position_billboard_fix_recall = position_billboard_fix_recall.drop('billboard', axis=1)
position_billboard_fix_recall.tail()

#munge the recognition df around
current_subject_recognition = recognition_clean_df[recognition_clean_df['Subject']==subjs[i]]
columns_to_drop = [col for col in current_subject_recognition.columns if col.endswith('_c') or col.endswith('_a')]
current_subject_recognition = current_subject_recognition.drop(['Subject'], axis= 1)
current_subject_recognition = current_subject_recognition.drop(columns_to_drop, axis=1)
current_subject_recognition = (current_subject_recognition.T)  
current_subject_recognition['billboard'] = current_subject_recognition.index
current_subject_recognition.rename( columns={i :'recognition'}, inplace=True )
current_subject_recognition.head()

# set_column1 = set(position_billboard_fix_recall['billboard_image_at_position_text'])
# set_column2 = set(current_subject_recognition['billboard'])

# # Find differences between the two sets
# differences_1_to_2 = set_column1 - set_column2
# differences_2_to_1 = set_column2 - set_column1

# # Check if there are any differences
# if not differences_1_to_2 and not differences_2_to_1:
#      print("The columns contain the same names.")
# else:
#     print("Differences from column 1 to 2:", differences_1_to_2)
#     print("Differences from column 2 to 1:", differences_2_to_1)
    
# #merge the fixation-recall and recognition df
# position_billboard_fix_recall_recognition = position_billboard_fix_recall.merge(current_subject_recognition, left_on = 'billboard_image_at_position_text', right_on = 'billboard')
# position_billboard_fix_recall_recognition.head()
# position_billboard_fix_recall_recognition = position_billboard_fix_recall_recognition.drop('billboard', axis=1)    
# position_billboard_fix_recall_recognition['recall'] = position_billboard_fix_recall_recognition['recall'].astype(int)
# position_billboard_fix_recall_recognition.head()

,recognition,billboard
ad_brunch,0.0,ad_brunch
ad_burger,1.0,ad_burger
ad_coffee,0.0,ad_coffee
ad_pizza,0.0,ad_pizza
ad_furniture,0.0,ad_furniture


In [7]:
for i in tqdm(range(len(subjs))):
    curr_sub_fixation_file = '../data/00_gaze_data/fixation_billboards/'  + subjs[i] +'_fixations_billboards.csv'
    out_file_name = '../data/03_fixation_billboards_recall_recognition/'  + subjs[i] +'_fixations_billboards_recall_recognition_integrated.csv'

    curr_sub_fix_df = pd.read_csv(curr_sub_fixation_file)
    curr_sub_fix_df['billboard_image_at_position_text_attention'] = curr_sub_fix_df.loc[:, 'billboard_image_at_position_text']
    curr_sub_fix_df['billboard_image_at_position_text'] = curr_sub_fix_df['billboard_image_at_position_text'].replace(to_replace=r'moreatt_', value='', regex=True)
    curr_sub_fix_df['billboard_image_at_position_text'] = curr_sub_fix_df['billboard_image_at_position_text'].replace(to_replace=r'lessatt_', value='', regex=True)
    curr_sub_fix_df.tail()

    position_billboard_fix = curr_sub_fix_df[["position_index", "billboard_image_at_position_text", "billboard_image_at_position_text_attention", "fix_bill", "billboard#_fixated_count", "billboard#_duration"]]
    position_billboard_fix.tail()

    current_subject_recall = memory_df[memory_df['subject']==subjs[i]]
    current_subject_recall = current_subject_recall.drop('subject', axis= 1)
    current_subject_recall = current_subject_recall.drop('condition', axis= 1)
    current_subject_recall = current_subject_recall.T
    current_subject_recall['billboard'] = current_subject_recall.index
    current_subject_recall.rename( columns={i :'recall'}, inplace=True )
    current_subject_recall = current_subject_recall.reset_index(drop=True)
    current_subject_recall.tail()

    # #merge fixation and recall df
    position_billboard_fix_recall = position_billboard_fix.merge(current_subject_recall, left_on = 'billboard_image_at_position_text', right_on = 'billboard')
    position_billboard_fix_recall = position_billboard_fix_recall.drop('billboard', axis=1)
    position_billboard_fix_recall.tail()

    #munge the recognition df around
    current_subject_recognition = recognition_clean_df[recognition_clean_df['Subject']==subjs[i]]
    columns_to_drop = [col for col in current_subject_recognition.columns if col.endswith('_c') or col.endswith('_a')]
    current_subject_recognition = current_subject_recognition.drop(['Subject'], axis= 1)
    current_subject_recognition = current_subject_recognition.drop(columns_to_drop, axis=1)
    current_subject_recognition = (current_subject_recognition.T)  
    current_subject_recognition['billboard'] = current_subject_recognition.index
    current_subject_recognition.columns = ['recognition', 'billboard']
    current_subject_recognition.head()

    #merge the fixation-recall and recognition df
    position_billboard_fix_recall_recognition = position_billboard_fix_recall.merge(current_subject_recognition, left_on = 'billboard_image_at_position_text', right_on = 'billboard')
    position_billboard_fix_recall_recognition.head()
    position_billboard_fix_recall_recognition = position_billboard_fix_recall_recognition.drop('billboard', axis=1)    
    position_billboard_fix_recall_recognition['recall'] = position_billboard_fix_recall_recognition['recall'].astype(int)
    position_billboard_fix_recall_recognition.head()

    #save to file
    position_billboard_fix_recall_recognition.to_csv(out_file_name)
    #position_billboard_fix_recall.to_csv(out_file_name)
    #print('---')
    
#print('done')

100%|██████████████████████████████████████████| 43/43 [00:00<00:00, 186.10it/s]
